In [1]:
"""
Causal GNN for Legal Judgment Prediction — QA-Pair Input (Track E)
===================================================================
STABLE VERSION v3 — XLNet + BiGRU Variant
==========================================

Model substitutions over the InLegalBERT+BiLSTM baseline:
  ★ Encoder  : law-ai/InLegalBERT  →  xlnet-base-cased
       - XLNet uses permutation-based autoregressive pretraining
       - Captures long-range bidirectional context without [CLS] pooling
       - last_hidden_state mean-pool used (no pooler_output in XLNet)
       - Requires attention_mask only (no token_type_ids by default)
  ★ Sequence : BiLSTM              →  BiGRU
       - Lighter gating (2 gates vs 3 in LSTM) → less overfit on small graphs
       - Replaces the CausalMP message-passing aggregation with a
         BiGRU that sequentially processes the 6 role-node sequence,
         capturing ordered legal narrative flow (FAC→ISSUE→ARG→…→RPC)
       - Hidden dim halved per direction so total stays = HIDDEN

All 8 v3 stability fixes retained:
  ① SWA evaluated only after ≥10 snapshots  (SWA_EVAL_AFTER = 40)
  ② Mixup always active throughout all epochs
  ③ ReduceLROnPlateau active even during SWA phase
  ④ best_state saves SWA model state dict
  ⑤ Intermediate SWA BN update every 10 epochs
  ⑥ Early stopping on clean metric timeline
  ⑦ Gradient clipping on base model only
  ⑧ Linear LR warmup for first 5 epochs
"""

# ──────────────────────────────────────────────────────────────
# 0.  Imports & reproducibility
# ──────────────────────────────────────────────────────────────
import json, random, math, warnings
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from transformers import AutoTokenizer, AutoModel          # XLNet via AutoModel
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, matthews_corrcoef,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score,
)
from sklearn.model_selection import StratifiedShuffleSplit

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ──────────────────────────────────────────────────────────────
# 1.  Constants
# ──────────────────────────────────────────────────────────────
ROLES     = ["FAC", "ISSUE", "ARG_P", "ANALYSIS", "RATIO", "RPC"]
ROLE2IDX  = {r: i for i, r in enumerate(ROLES)}
NUM_ROLES = len(ROLES)

# ★ XLNet hidden size = 768 (same as BERT-base)
EMB_DIM      = 768
MAX_LEN      = 256
HIDDEN       = 256          # total BiGRU output dim (128 per direction × 2)
BIGRU_LAYERS = 2            # stacked BiGRU depth
NUM_MP       = 2            # causal message-passing rounds (post-BiGRU)
DROPOUT_P    = 0.50
DROPOUT_C    = 0.35
BATCH        = 16
EPOCHS       = 80
LR           = 1e-5
WD           = 5e-4
CLIP         = 0.5
TRAIN_R      = 0.80
ES_PAT       = 20
LABEL_SM     = 0.05
MIXUP_A      = 0.3
WARMUP_EP    = 5

# SWA
SWA_START      = 30
SWA_LR         = 3e-5
SWA_EVAL_AFTER = 40
SWA_BN_EVERY   = 10

# ★ XLNet model name
XLNET_MODEL = "xlnet-base-cased"

SIGNAL_MAP = {
    "FAVORS_PETITIONER": +1.0,
    "NEUTRAL":            0.0,
    "FAVORS_RESPONDENT": -1.0,
}

PLOT_DIR = "."

# ──────────────────────────────────────────────────────────────
# 2.  Load & group QA pairs  (unchanged)
# ──────────────────────────────────────────────────────────────
def load_qa_jsonl(path: str) -> dict:
    docs = defaultdict(lambda: {"label": None, "roles": defaultdict(list)})
    with open(path, "r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"  [WARN] line {lineno} skipped — {e}")
                continue

            doc_id = rec.get("doc_id") or rec.get("id", "").rsplit("_Q", 1)[0]
            role   = rec.get("role", "FAC")
            label  = rec.get("label")
            signal = rec.get("signal", "NEUTRAL")
            if isinstance(signal, dict):
                signal = signal.get("label", "NEUTRAL")

            docs[doc_id]["roles"][role].append({
                "question": rec.get("question", ""),
                "answer":   rec.get("answer",   ""),
                "signal":   signal,
            })
            if docs[doc_id]["label"] is None and label is not None:
                docs[doc_id]["label"] = int(label)

    docs = {k: v for k, v in docs.items() if v["label"] is not None}
    print(f"Loaded {len(docs)} labelled documents from '{path}'")
    return docs


# ──────────────────────────────────────────────────────────────
# 3.  ★ XLNet Embedder  (replaces InLegalBERTEmbedder)
# ──────────────────────────────────────────────────────────────
class XLNetEmbedder:
    """
    Wraps xlnet-base-cased for sentence-level mean-pool embeddings.

    Key XLNet differences vs BERT:
      • No [CLS] token with a dedicated pooler — we mean-pool last_hidden_state
      • Permutation LM: each token attends to all others in permuted order
        → richer bidirectional context captured in hidden states
      • token_type_ids not required for single-segment input
      • Padding is left-padded by default in XLNet tokenizer;
        we keep right-padding (truncation_side="right") for consistency
    """
    def __init__(self, model_name: str = XLNET_MODEL):
        print(f"Loading {model_name} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            padding_side="right",       # consistent with BERT-style training
        )
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval().to(DEVICE)
        print(f"XLNet loaded — hidden size: {self.model.config.hidden_size}")

    @torch.no_grad()
    def _mean_pool(self, h, mask):
        """Mean-pool over non-padding positions."""
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

    @torch.no_grad()
    def embed(self, texts, batch_size=16):
        out     = []
        cleaned = [t if t.strip() else "[SEP]" for t in texts]   # XLNet uses [SEP] not [PAD]
        for s in range(0, len(cleaned), batch_size):
            batch = cleaned[s: s + batch_size]
            enc   = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=MAX_LEN,
                return_tensors="pt",
            )
            iids  = enc["input_ids"].to(DEVICE)
            amask = enc["attention_mask"].to(DEVICE)
            # XLNet does not require token_type_ids for single-sequence input
            h = self.model(input_ids=iids, attention_mask=amask).last_hidden_state
            out.append(self._mean_pool(h, amask).cpu().numpy())
            print(f"  Embedded {min(s+batch_size, len(cleaned))}/{len(cleaned)}", end="\r")
        print()
        return np.vstack(out)

    def build_doc_embeddings(self, docs: dict) -> dict:
        all_texts, all_meta = [], []
        for doc_id, doc in docs.items():
            for role, qas in doc["roles"].items():
                for qa in qas:
                    q = qa["question"].strip()
                    a = qa["answer"].strip()
                    all_texts.append(f"[Q] {q} [A] {a}" if q else f"[A] {a}")
                    all_meta.append((doc_id, role,
                                     SIGNAL_MAP.get(qa["signal"], 0.0)))

        print(f"\nEmbedding {len(all_texts)} QA pairs across {len(docs)} docs ...")
        embs = self.embed(all_texts)

        role_embs    = defaultdict(lambda: defaultdict(list))
        role_signals = defaultdict(lambda: defaultdict(list))
        for i, (doc_id, role, sig) in enumerate(all_meta):
            role_embs[doc_id][role].append(embs[i])
            role_signals[doc_id][role].append(sig)

        for doc_id, doc in docs.items():
            node_feats = np.zeros((NUM_ROLES, EMB_DIM), dtype=np.float32)
            for role_name, ridx in ROLE2IDX.items():
                vecs = role_embs[doc_id].get(role_name, [])
                sigs = role_signals[doc_id].get(role_name, [])
                if vecs:
                    h_r = np.mean(vecs, axis=0)
                    es  = float(np.mean(sigs))
                    node_feats[ridx] = h_r * (1.0 + es)
            doc["node_feats"] = node_feats

        print("XLNet embedding & entailment gating complete.")
        return docs


# ──────────────────────────────────────────────────────────────
# 4.  Dataset  (unchanged)
# ──────────────────────────────────────────────────────────────
class LegalQADataset(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        doc_id, feats, label = self.items[idx]
        return (torch.tensor(feats,  dtype=torch.float32),
                torch.tensor(label,  dtype=torch.float32),
                doc_id)

def collate(batch):
    feats, labels, ids = zip(*batch)
    return torch.stack(feats), torch.stack(labels), list(ids)

def make_items(docs):
    return [(doc_id, doc["node_feats"], doc["label"])
            for doc_id, doc in docs.items()
            if "node_feats" in doc and doc["label"] is not None]


# ──────────────────────────────────────────────────────────────
# 5.  Stratified split  (unchanged)
# ──────────────────────────────────────────────────────────────
def stratified_split(items, train_ratio=TRAIN_R, seed=SEED):
    labels = np.array([it[2] for it in items])
    sss    = StratifiedShuffleSplit(n_splits=1,
                                    test_size=1 - train_ratio,
                                    random_state=seed)
    train_idx, test_idx = next(sss.split(np.zeros(len(labels)), labels))
    return train_idx.tolist(), test_idx.tolist()


# ──────────────────────────────────────────────────────────────
# 6.  ★ Model components  (BiGRU replaces BiLSTM inside CausalMP)
# ──────────────────────────────────────────────────────────────

class CausalAdjacency(nn.Module):
    """Learnable upper-triangular causal adjacency matrix (unchanged)."""
    def __init__(self, n=NUM_ROLES):
        super().__init__()
        self.W_raw = nn.Parameter(torch.randn(n, n) * 0.1)
        mask = torch.triu(torch.ones(n, n), diagonal=1)
        self.register_buffer("mask", mask)
    def forward(self):
        return torch.sigmoid(self.W_raw) * self.mask


class BiGRUContextualiser(nn.Module):
    """
    ★ BiGRU module that replaces the BiLSTM role-sequence encoder.

    Takes the 6-node role sequence (B, NUM_ROLES, hidden_dim) and
    returns a contextualised representation of the same shape.

    Why BiGRU over BiLSTM here:
      - Legal role sequence is short (N=6): GRU's 2-gate design
        suffices; the extra cell state in LSTM adds parameters
        without benefit on such short sequences.
      - Fewer parameters → less overfit on small legal datasets.
      - Forward + backward passes capture both
        "FAC informs ISSUE" and "RPC reflects RATIO" directions.

    Architecture:
      input  (B, 6, hidden_dim)
      → BiGRU(hidden_dim//2, num_layers, bidirectional=True)
         [hidden per dir = HIDDEN//2, total output = HIDDEN]
      → LayerNorm + residual
      output (B, 6, hidden_dim)
    """
    def __init__(self, hidden_dim=HIDDEN, num_layers=BIGRU_LAYERS,
                 dropout=0.25):
        super().__init__()
        assert hidden_dim % 2 == 0, "hidden_dim must be even for BiGRU"
        self.bigru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim // 2,   # × 2 directions = hidden_dim total
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.ln  = nn.LayerNorm(hidden_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, H):
        # H: (B, NUM_ROLES, hidden_dim)
        out, _ = self.bigru(H)             # (B, NUM_ROLES, hidden_dim)
        return self.ln(H + self.drop(out)) # residual + LayerNorm


class CausalMP(nn.Module):
    """
    Causal message-passing round.
    Same structure as v3; operates on already-BiGRU-contextualised features.
    """
    def __init__(self, dim):
        super().__init__()
        self.lin = nn.Linear(dim, dim)
        self.act = nn.GELU()
        self.ln  = nn.LayerNorm(dim)
    def forward(self, H, W):
        Ht = self.act(self.lin(H))
        M  = torch.einsum("ij,bid->bjd", W, Ht)
        return self.ln(H + M)


class CausalGNN(nn.Module):
    """
    ★ CausalGNN with XLNet embeddings + BiGRU contextualiser.

    Pipeline per forward pass:
      x (B, 6, 768)
      → proj     : Linear + LayerNorm + GELU + Dropout  → (B, 6, HIDDEN)
      → bigru    : BiGRU contextualiser                 → (B, 6, HIDDEN)
      → causal_mp: NUM_MP rounds of causal message-pass → (B, 6, HIDDEN)
      → pool     : mean + max concat                    → (B, 2*HIDDEN)
      → cls      : MLP classifier                       → (B,)
    """
    def __init__(self, emb_dim=EMB_DIM, hidden_dim=HIDDEN,
                 num_rounds=NUM_MP,
                 bigru_layers=BIGRU_LAYERS,
                 dropout_p=DROPOUT_P, dropout_c=DROPOUT_C):
        super().__init__()
        self.adj = CausalAdjacency(NUM_ROLES)

        # Project XLNet 768-d embeddings to HIDDEN
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_p),
        )

        # ★ BiGRU contextualiser over role sequence
        self.bigru = BiGRUContextualiser(
            hidden_dim=hidden_dim,
            num_layers=bigru_layers,
            dropout=0.25,
        )

        # Causal message-passing rounds (operate on BiGRU output)
        self.mp = nn.ModuleList([CausalMP(hidden_dim) for _ in range(num_rounds)])

        # Classifier head
        self.cls = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_c),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout_c / 2),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x, return_adj=False):
        # x: (B, NUM_ROLES, EMB_DIM)
        W = self.adj()

        H = self.proj(x)           # (B, N, HIDDEN)
        H = self.bigru(H)          # ★ BiGRU contextualisation
        for mp in self.mp:
            H = mp(H, W)           # causal message passing

        mean_p = H.mean(1)
        max_p  = H.max(1).values
        pooled = torch.cat([mean_p, max_p], -1)
        logits = self.cls(pooled).squeeze(-1)
        return (logits, W) if return_adj else logits

    def get_adjacency(self):
        with torch.no_grad():
            return self.adj().cpu().numpy()


# ──────────────────────────────────────────────────────────────
# 7.  Label-smoothing BCE  (unchanged)
# ──────────────────────────────────────────────────────────────
class LabelSmoothBCE(nn.Module):
    def __init__(self, pos_weight, epsilon=LABEL_SM):
        super().__init__()
        self.eps = epsilon
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    def forward(self, logits, targets):
        soft = targets * (1 - self.eps) + (1 - targets) * self.eps
        return self.bce(logits, soft)


# ──────────────────────────────────────────────────────────────
# 8.  Mixup — always active  (unchanged)
# ──────────────────────────────────────────────────────────────
def mixup_batch(feats, labels, alpha=MIXUP_A):
    if alpha <= 0:
        return feats, labels
    lam = np.random.beta(alpha, alpha)
    B   = feats.size(0)
    idx = torch.randperm(B, device=feats.device)
    return (lam * feats + (1 - lam) * feats[idx],
            lam * labels + (1 - lam) * labels[idx])


# ──────────────────────────────────────────────────────────────
# 9.  Metrics  (unchanged)
# ──────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    wf1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    mf1 = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.5
    return {"acc": acc, "wf1": wf1, "mf1": mf1, "mcc": mcc, "auc": auc}


# ──────────────────────────────────────────────────────────────
# 10. Evaluate helper  (unchanged)
# ──────────────────────────────────────────────────────────────
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    all_p, all_y, all_prob = [], [], []
    with torch.no_grad():
        for feats, labels, _ in loader:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            logits = model(feats)
            total_loss += loss_fn(logits, labels).item()
            probs = torch.sigmoid(logits).cpu().numpy()
            all_prob.extend(probs)
            all_p.extend((probs >= 0.5).astype(int))
            all_y.extend(labels.cpu().long().numpy())
    m = compute_metrics(np.array(all_y), np.array(all_p), np.array(all_prob))
    return total_loss / len(loader), m


# ──────────────────────────────────────────────────────────────
# 11. pos_weight  (unchanged)
# ──────────────────────────────────────────────────────────────
def get_pos_weight(items):
    labels = [it[2] for it in items]
    n_pos  = sum(labels)
    n_neg  = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return torch.tensor(1.0)
    return torch.tensor(n_neg / n_pos, dtype=torch.float32)


# ──────────────────────────────────────────────────────────────
# 12. WarmupThenPlateau scheduler  (unchanged)
# ──────────────────────────────────────────────────────────────
class WarmupThenPlateau:
    def __init__(self, optimizer, warmup_epochs, base_lr, plateau_scheduler):
        self.opt       = optimizer
        self.warmup_ep = warmup_epochs
        self.base_lr   = base_lr
        self.plateau   = plateau_scheduler
        self._ep       = 0

    def step(self, val_loss=None):
        self._ep += 1
        if self._ep <= self.warmup_ep:
            lr = self.base_lr * (self._ep / self.warmup_ep)
            for pg in self.opt.param_groups:
                pg["lr"] = lr
        elif val_loss is not None:
            self.plateau.step(val_loss)

    def get_lr(self):
        return self.opt.param_groups[0]["lr"]


# ──────────────────────────────────────────────────────────────
# 13. Training loop — v3 all 8 fixes  (unchanged logic)
# ──────────────────────────────────────────────────────────────
def train(model, train_loader, test_loader,
          pos_weight, epochs=EPOCHS, lr=LR):
    model.to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=lr / 10, weight_decay=WD)
    _plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=6, min_lr=1e-6, verbose=True
    )
    scheduler  = WarmupThenPlateau(opt, WARMUP_EP, lr, _plateau)
    swa_model  = AveragedModel(model)
    swa_sch    = SWALR(opt, swa_lr=SWA_LR, anneal_epochs=5)
    swa_active = False
    loss_fn    = LabelSmoothBCE(pos_weight.to(DEVICE))

    history = {k: [] for k in
               ["train_loss", "test_loss",
                "acc", "wf1", "mf1", "auc", "mcc", "lr", "eval_source"]}

    best_wf1, best_state, best_is_swa = 0.0, None, False
    pat_ctr, swa_snapshots = 0, 0

    for ep in range(1, epochs + 1):

        # ── SWA activation ────────────────────────────────────
        if ep == SWA_START and not swa_active:
            swa_active = True
            print(f"\n  [SWA] Snapshot collection starts at epoch {ep}")
            print(f"  [SWA] Evaluation switches to SWA model at epoch {SWA_EVAL_AFTER}")

        # ── Train base model ───────────────────────────────────
        model.train()
        total = 0.0
        for feats, labels, _ in train_loader:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            feats, labels = mixup_batch(feats, labels, MIXUP_A)   # ② always on
            opt.zero_grad()
            loss = loss_fn(model(feats), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CLIP)    # ⑦
            opt.step()
            total += loss.item()
        tr_loss = total / len(train_loader)

        # ── SWA snapshot ──────────────────────────────────────
        if swa_active:
            swa_model.update_parameters(model)
            swa_snapshots += 1
            swa_sch.step()
            if swa_snapshots % SWA_BN_EVERY == 0:                 # ⑤
                update_bn(train_loader, swa_model, device=DEVICE)

        # ── ① Eval model selection ────────────────────────────
        use_swa_eval = swa_active and (ep >= SWA_EVAL_AFTER)
        eval_model   = swa_model if use_swa_eval else model
        eval_source  = "swa" if use_swa_eval else "base"

        te_loss, m = evaluate(eval_model, test_loader, loss_fn)
        current_lr = scheduler.get_lr()

        scheduler.step(val_loss=te_loss)   # ③ always active

        for k in ["acc", "wf1", "mf1", "auc", "mcc"]:
            history[k].append(m[k])
        history["train_loss"].append(tr_loss)
        history["test_loss"].append(te_loss)
        history["lr"].append(current_lr)
        history["eval_source"].append(eval_source)

        flag    = "★" if m["wf1"] > best_wf1 else " "
        src_tag = f"[{eval_source.upper():4s}]"
        print(f"Ep {ep:3d}/{epochs} {flag} {src_tag} | lr {current_lr:.2e} | "
              f"TrL {tr_loss:.4f} | TeL {te_loss:.4f} | "
              f"Acc {m['acc']:.4f} | wF1 {m['wf1']:.4f} | "
              f"AUC {m['auc']:.4f} | MCC {m['mcc']:.4f}")

        if m["wf1"] > best_wf1:
            best_wf1    = m["wf1"]
            best_is_swa = use_swa_eval
            src_dict    = swa_model.state_dict() if use_swa_eval \
                          else model.state_dict()
            best_state  = {k: v.clone() for k, v in src_dict.items()}  # ④
            pat_ctr     = 0
        else:
            pat_ctr += 1
            if pat_ctr >= ES_PAT:
                print(f"\nEarly stop at epoch {ep}  (patience={ES_PAT})")
                break

    # ── Final BN update ───────────────────────────────────────
    if swa_active:
        print("\nFinal SWA BN statistics update ...")
        update_bn(train_loader, swa_model, device=DEVICE)

    # ── ④ Restore best checkpoint ─────────────────────────────
    if best_state:
        if best_is_swa:
            swa_model.load_state_dict(best_state)
            print(f"Restored best SWA model  (wF1 = {best_wf1:.4f})")
        else:
            model.load_state_dict(best_state)
            print(f"Restored best base model  (wF1 = {best_wf1:.4f})")

    return history, (swa_model if swa_active else model)


# ──────────────────────────────────────────────────────────────
# 14. Full inference  (unchanged)
# ──────────────────────────────────────────────────────────────
def full_inference(model, loader):
    model.eval()
    all_true, all_pred, all_prob, all_ids = [], [], [], []
    with torch.no_grad():
        for feats, labels, ids in loader:
            feats = feats.to(DEVICE)
            try:
                logits, _ = model(feats, return_adj=True)
            except TypeError:
                logits = model(feats)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_prob.extend(probs)
            all_pred.extend((probs >= 0.5).astype(int))
            all_true.extend(labels.long().numpy())
            all_ids.extend(ids)
    return (np.array(all_true), np.array(all_pred),
            np.array(all_prob), all_ids)


# ──────────────────────────────────────────────────────────────
# 15. Plots  (titles updated to reflect XLNet + BiGRU)
# ──────────────────────────────────────────────────────────────
PALETTE = {
    "train": "#4C72B0", "val": "#DD8452",
    "pos"  : "#55A868", "neg": "#C44E52",
    "bg"   : "#F8F9FA",
}

def _savefig(name):
    path = f"{PLOT_DIR}/{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")

def _ema(x, alpha=0.2):
    out, s = [], x[0]
    for v in x:
        s = alpha * v + (1 - alpha) * s
        out.append(s)
    return out


def plot_losses(history):
    fig, ax = plt.subplots(figsize=(11, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    eps = range(1, len(history["train_loss"]) + 1)

    ax.plot(eps, history["train_loss"], color=PALETTE["train"], lw=1, alpha=0.35)
    ax.plot(eps, _ema(history["train_loss"]), color=PALETTE["train"],
            lw=2.2, label="Train loss (EMA)")
    ax.plot(eps, history["test_loss"], color=PALETTE["val"],
            lw=1, alpha=0.35, linestyle="--")
    ax.plot(eps, _ema(history["test_loss"]), color=PALETTE["val"],
            lw=2.2, linestyle="--", label="Val loss (EMA)")

    n = len(history["train_loss"])
    if SWA_START <= n:
        ax.axvspan(SWA_START, min(SWA_EVAL_AFTER, n),
                   alpha=0.08, color="gray",
                   label="SWA snapshot phase\n(base model evaluated)")
        ax.axvline(SWA_START, color="#888", lw=1.2, linestyle=":",
                   label=f"SWA start (ep {SWA_START})")
    if SWA_EVAL_AFTER <= n:
        ax.axvline(SWA_EVAL_AFTER, color="#4a9", lw=1.5, linestyle="--",
                   label=f"SWA eval active (ep {SWA_EVAL_AFTER})")

    gap = abs(history["train_loss"][-1] - history["test_loss"][-1])
    ax.annotate(f"Gap: {gap:.4f}",
                xy=(len(eps), (history["train_loss"][-1] +
                               history["test_loss"][-1]) / 2),
                xytext=(-70, 0), textcoords="offset points",
                fontsize=9, color="#555",
                arrowprops=dict(arrowstyle="->", color="#999"))
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("P1 — Training vs Validation Loss  [XLNet + BiGRU — v3 Stable]",
                 fontweight="bold")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.annotate(f'{history["train_loss"][-1]:.4f}',
                xy=(len(eps), history["train_loss"][-1]),
                xytext=(-35, 8), textcoords="offset points",
                color=PALETTE["train"], fontsize=8)
    ax.annotate(f'{history["test_loss"][-1]:.4f}',
                xy=(len(eps), history["test_loss"][-1]),
                xytext=(-35, -14), textcoords="offset points",
                color=PALETTE["val"], fontsize=8)
    _savefig("P1_loss_curves")


def plot_metrics(history):
    metrics = [("acc", "Accuracy", PALETTE["train"]),
               ("wf1", "Weighted F1", PALETTE["val"]),
               ("auc", "AUC-ROC", PALETTE["pos"]),
               ("mcc", "MCC", PALETTE["neg"])]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), facecolor=PALETTE["bg"])
    fig.suptitle("P2 — Metrics over Epochs  [XLNet + BiGRU — v3]",
                 fontweight="bold", fontsize=13)
    axes = axes.flatten()
    eps  = range(1, len(history["acc"]) + 1)
    for ax, (key, title, color) in zip(axes, metrics):
        ax.set_facecolor(PALETTE["bg"])
        ax.plot(eps, history[key], color=color, lw=1, alpha=0.4)
        ax.plot(eps, _ema(history[key], 0.25), color=color, lw=2.2)
        ax.axhline(max(history[key]), color=color, lw=1, linestyle=":", alpha=0.6)
        n = len(history[key])
        if SWA_START <= n:
            ax.axvspan(SWA_START, min(SWA_EVAL_AFTER, n), alpha=0.06, color="gray")
            ax.axvline(SWA_START, color="#888", lw=1, linestyle=":")
        if SWA_EVAL_AFTER <= n:
            ax.axvline(SWA_EVAL_AFTER, color="#4a9", lw=1.2, linestyle="--")
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.set_ylabel(title)
        ax.grid(alpha=0.3)
        best_ep = int(np.argmax(history[key])) + 1
        ax.annotate(f"Best: {max(history[key]):.4f} @ ep {best_ep}",
                    xy=(best_ep, max(history[key])),
                    xytext=(10, -14), textcoords="offset points",
                    fontsize=8, color=color)
    _savefig("P2_metric_curves")


def plot_confusion(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    labels_str = ["Rejected (0)", "Affirmed (1)"]
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels_str, yticklabels=labels_str,
                linewidths=0.5, linecolor="#cccccc", ax=ax,
                annot_kws={"size": 14, "weight": "bold"})
    ax.set_xlabel("Predicted", fontsize=11); ax.set_ylabel("Actual", fontsize=11)
    ax.set_title("P3 — Confusion Matrix  [XLNet + BiGRU]",
                 fontweight="bold", fontsize=12)
    for i, (row, _) in enumerate(zip(cm, labels_str)):
        pct = row[i] / row.sum() * 100 if row.sum() else 0
        ax.text(i + 0.5, i + 0.7, f"({pct:.1f}%)",
                ha="center", va="center", fontsize=9, color="#333333")
    _savefig("P3_confusion_matrix")


def plot_roc(y_true, y_prob):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val      = roc_auc_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.plot(fpr, tpr, color=PALETTE["train"], lw=2,
            label=f"ROC (AUC = {auc_val:.4f})")
    ax.fill_between(fpr, tpr, alpha=0.10, color=PALETTE["train"])
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("P4 — ROC Curve  [XLNet + BiGRU]", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P4_roc_curve")


def plot_pr(y_true, y_prob):
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.plot(rec, prec, color=PALETTE["pos"], lw=2, label=f"PR (AP = {ap:.4f})")
    ax.fill_between(rec, prec, alpha=0.10, color=PALETTE["pos"])
    baseline = y_true.mean()
    ax.axhline(baseline, color="gray", lw=1, linestyle="--",
               label=f"Baseline ({baseline:.2f})")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("P5 — Precision-Recall Curve  [XLNet + BiGRU]", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P5_pr_curve")


def plot_cls_report(y_true, y_pred):
    report = classification_report(y_true, y_pred,
                                   target_names=["Rejected", "Affirmed"],
                                   output_dict=True)
    rows = ["Rejected", "Affirmed", "macro avg", "weighted avg"]
    cols = ["precision", "recall", "f1-score", "support"]
    data = [[report[r][c] for c in cols] for r in rows]
    fig, ax = plt.subplots(figsize=(9, 3.5), facecolor=PALETTE["bg"])
    ax.axis("off")
    tbl = ax.table(
        cellText=[[f"{v:.4f}" if isinstance(v, float) else str(int(v))
                   for v in row] for row in data],
        rowLabels=rows, colLabels=[c.title() for c in cols],
        cellLoc="center", loc="center",
    )
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 2.0)
    for j in range(len(cols)):
        tbl[(0, j)].set_facecolor("#4C72B0")
        tbl[(0, j)].set_text_props(color="white", fontweight="bold")
    for i in range(1, len(rows) + 1):
        tbl[(i, -1)].set_facecolor("#E8EDF5")
        tbl[(i, -1)].set_text_props(fontweight="bold")
    ax.set_title("P6 — Classification Report  [XLNet + BiGRU]",
                 fontweight="bold", fontsize=12, pad=12)
    _savefig("P6_classification_report")
    print("\n" + "="*60)
    print("CLASSIFICATION REPORT  [XLNet + BiGRU]")
    print("="*60)
    print(classification_report(y_true, y_pred,
                                target_names=["Rejected", "Affirmed"]))


def plot_adjacency(model):
    base = model.module if hasattr(model, "module") else model
    W    = base.get_adjacency()
    fig, ax = plt.subplots(figsize=(7, 6), facecolor=PALETTE["bg"])
    sns.heatmap(W, annot=True, fmt=".3f", cmap="YlOrRd",
                xticklabels=ROLES, yticklabels=ROLES,
                linewidths=0.4, linecolor="#cccccc",
                vmin=0, vmax=1, ax=ax, annot_kws={"size": 9})
    ax.set_title("P7 — Learned Causal Adjacency  [XLNet + BiGRU]",
                 fontweight="bold")
    ax.set_xlabel("Target Role"); ax.set_ylabel("Source Role")
    _savefig("P7_adjacency_matrix")
    edges = sorted(
        [(W[i, j], ROLES[i], ROLES[j])
         for i in range(NUM_ROLES) for j in range(NUM_ROLES)
         if i < j and W[i, j] > 0.05], reverse=True
    )
    print("\nTop-5 Causal Edges:")
    for w, s, t in edges[:5]:
        print(f"  {s:>10} → {t:<10}  weight: {w:.4f}")


def plot_transplant(flip_rates: dict):
    roles  = list(flip_rates.keys())
    rates  = [flip_rates[r] for r in roles]
    colors = [PALETTE["neg"] if r == max(flip_rates, key=flip_rates.get)
              else PALETTE["train"] for r in roles]
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    bars = ax.bar(roles, rates, color=colors, edgecolor="white", width=0.55)
    ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
    ax.set_ylim(0, max(rates) * 1.25 if max(rates) > 0 else 0.5)
    ax.set_ylabel("Flip Rate (Affirm → Reject)")
    ax.set_title("P8 — Transplant Experiment  [XLNet + BiGRU]", fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    decisive = max(flip_rates, key=flip_rates.get)
    ax.text(0.5, 0.92, f"Most decisive: {decisive}",
            ha="center", va="center", transform=ax.transAxes,
            fontsize=10, style="italic", color=PALETTE["neg"])
    _savefig("P8_transplant_flip_rates")


def plot_prob_dist(y_true, y_prob):
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.hist(y_prob[y_true == 0], bins=25, alpha=0.65,
            color=PALETTE["neg"], label="True Rejected (0)", edgecolor="white")
    ax.hist(y_prob[y_true == 1], bins=25, alpha=0.65,
            color=PALETTE["pos"], label="True Affirmed (1)", edgecolor="white")
    ax.axvline(0.5, color="black", lw=1.5, linestyle="--", label="Threshold 0.5")
    ax.set_xlabel("P(Affirmed)"); ax.set_ylabel("Count")
    ax.set_title("P9 — Prediction Probability Distribution  [XLNet + BiGRU]",
                 fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P9_prob_distribution")


# ──────────────────────────────────────────────────────────────
# 16. Transplant Experiment  (unchanged logic)
# ──────────────────────────────────────────────────────────────
def transplant_experiment(model, docs, n_pairs=200):
    model.eval()
    affirmed, rejected = [], []
    with torch.no_grad():
        for doc_id, doc in docs.items():
            if "node_feats" not in doc:
                continue
            feats = torch.tensor(doc["node_feats"],
                                 dtype=torch.float32).unsqueeze(0).to(DEVICE)
            try:
                logits, _ = model(feats, return_adj=True)
            except TypeError:
                logits = model(feats)
            prob = torch.sigmoid(logits).item()
            pred = int(prob >= 0.5)
            if pred == doc["label"]:
                (affirmed if doc["label"] == 1 else rejected).append(doc)

    print(f"\nTransplant pool — Affirmed: {len(affirmed)}, Rejected: {len(rejected)}")
    if not affirmed or not rejected:
        print("[WARN] Transplant skipped — insufficient correctly-predicted cases.")
        return {r: 0.0 for r in ROLES}

    flip_counts = {r: 0 for r in ROLES}
    total_pairs = {r: 0 for r in ROLES}
    random.shuffle(affirmed); random.shuffle(rejected)

    for doc_A in affirmed[:n_pairs]:
        doc_B = random.choice(rejected)
        for ridx, role in enumerate(ROLES):
            emb_A  = doc_A["node_feats"].copy()
            f_orig = torch.tensor(emb_A, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                try:
                    logits, _ = model(f_orig, return_adj=True)
                except TypeError:
                    logits = model(f_orig)
                p_orig = int(torch.sigmoid(logits).item() >= 0.5)

            transplanted       = emb_A.copy()
            transplanted[ridx] = doc_B["node_feats"][ridx]
            f_new = torch.tensor(transplanted,
                                 dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                try:
                    logits, _ = model(f_new, return_adj=True)
                except TypeError:
                    logits = model(f_new)
                p_new = int(torch.sigmoid(logits).item() >= 0.5)

            if p_orig == 1 and p_new == 0:
                flip_counts[role] += 1
            total_pairs[role] += 1

    flip_rates = {r: flip_counts[r] / max(total_pairs[r], 1) for r in ROLES}
    print("\n" + "="*55)
    print("TRANSPLANT — Decisive Role Map  (flip rate 1→0)")
    print("="*55)
    for role in ROLES:
        bar = "█" * int(flip_rates[role] * 40)
        print(f"  {role:<10}  {flip_rates[role]:.3f}  {bar}")
    decisive = max(flip_rates, key=flip_rates.get)
    print(f"\nMost decisive role: {decisive} ({flip_rates[decisive]:.3f})")
    return flip_rates


# ──────────────────────────────────────────────────────────────
# 17. Final summary
# ──────────────────────────────────────────────────────────────
def print_final_summary(y_true, y_pred, y_prob):
    m = compute_metrics(y_true, y_pred, y_prob)
    print("\n" + "="*60)
    print("FINAL TEST METRICS  [XLNet + BiGRU — v3]")
    print("="*60)
    print(f"  Accuracy    : {m['acc']:.4f}  {'✓' if m['acc']>=0.84 else '✗'}  (target ≥ 0.84)")
    print(f"  Weighted F1 : {m['wf1']:.4f}  {'✓' if m['wf1']>=0.85 else '✗'}  (target ≥ 0.85)")
    print(f"  Macro F1    : {m['mf1']:.4f}")
    print(f"  AUC-ROC     : {m['auc']:.4f}")
    print(f"  MCC         : {m['mcc']:.4f}")
    print("="*60)
    return m


# ──────────────────────────────────────────────────────────────
# 18. Main
# ──────────────────────────────────────────────────────────────
def main():
    DATA_PATH = "cjpe_100k_qa_flat.jsonl"

    print("\n" + "="*60)
    print("STAGE 1 — Loading QA pairs")
    print("="*60)
    docs = load_qa_jsonl(DATA_PATH)

    print("\n" + "="*60)
    print("STAGE 2-3 — XLNet Embedding + Track-E Entailment Gating")
    print("="*60)
    embedder = XLNetEmbedder(XLNET_MODEL)        # ★ XLNet
    docs     = embedder.build_doc_embeddings(docs)

    items   = make_items(docs)
    n_total = len(items)
    train_idx, test_idx = stratified_split(items, TRAIN_R)
    n_train, n_test = len(train_idx), len(test_idx)

    dataset  = LegalQADataset(items)
    train_ds = Subset(dataset, train_idx)
    test_ds  = Subset(dataset, test_idx)

    train_items = [items[i] for i in train_idx]
    pw = get_pos_weight(train_items)
    print(f"\nDataset — total: {n_total} | train: {n_train} | test: {n_test}")
    print(f"Pos-weight for BCE: {pw.item():.4f}")

    tr_labels = [items[i][2] for i in train_idx]
    te_labels = [items[i][2] for i in test_idx]
    print(f"Train  —  +: {sum(tr_labels)} | -: {n_train - sum(tr_labels)}")
    print(f"Test   —  +: {sum(te_labels)} | -: {n_test  - sum(te_labels)}")

    train_loader = DataLoader(train_ds, batch_size=BATCH,
                              shuffle=True,  collate_fn=collate)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH,
                              shuffle=False, collate_fn=collate)

    print("\n" + "="*60)
    print("STAGE 4-5 — Training Causal GNN  [XLNet + BiGRU — v3]")
    print("="*60)
    model = CausalGNN(                           # ★ BiGRU inside
        emb_dim=EMB_DIM, hidden_dim=HIDDEN,
        num_rounds=NUM_MP,
        bigru_layers=BIGRU_LAYERS,
        dropout_p=DROPOUT_P, dropout_c=DROPOUT_C,
    )
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {n_params:,}")
    print(f"\nKey hyperparameters:")
    print(f"  Encoder      : {XLNET_MODEL}  (XLNet)")
    print(f"  Seq model    : BiGRU  (layers={BIGRU_LAYERS}, "
          f"hidden/dir={HIDDEN//2}, total={HIDDEN})")
    print(f"  LR={LR}  WD={WD}  CLIP={CLIP}  WARMUP_EP={WARMUP_EP}")
    print(f"  HIDDEN={HIDDEN}  NUM_MP={NUM_MP}")
    print(f"  DROPOUT_P={DROPOUT_P}  DROPOUT_C={DROPOUT_C}")
    print(f"  SWA snapshot start = epoch {SWA_START}")
    print(f"  SWA eval active    = epoch {SWA_EVAL_AFTER}  (≥10 snapshots)")
    print(f"  SWA_BN_EVERY={SWA_BN_EVERY}  SWA_LR={SWA_LR}")
    print(f"  Mixup α={MIXUP_A} (always on)  Label smoothing ε={LABEL_SM}")

    history, final_model = train(model, train_loader, test_loader,
                                 pos_weight=pw, epochs=EPOCHS, lr=LR)

    torch.save(model.state_dict(), "causal_gnn_xlnet_bigru_v3.pt")
    print("Model saved → causal_gnn_xlnet_bigru_v3.pt")

    print("\n" + "="*60)
    print("STAGE 6 — Inference & Full Evaluation")
    print("="*60)
    y_true, y_pred, y_prob, _ = full_inference(final_model, test_loader)
    print_final_summary(y_true, y_pred, y_prob)

    print("\n" + "="*60)
    print("STAGE 7 — Cross-Case Transplant Experiment")
    print("="*60)
    flip_rates = transplant_experiment(final_model, docs, n_pairs=200)

    print("\n" + "="*60)
    print("STAGE 8 — Generating all diagnostic plots")
    print("="*60)
    plot_losses(history)
    plot_metrics(history)
    plot_confusion(y_true, y_pred)
    plot_roc(y_true, y_prob)
    plot_pr(y_true, y_prob)
    plot_cls_report(y_true, y_pred)
    plot_adjacency(final_model)
    plot_transplant(flip_rates)
    plot_prob_dist(y_true, y_prob)

    print("\n✓ All 9 plots saved.")
    for i, name in enumerate(
        ["P1_loss_curves", "P2_metric_curves",
         "P3_confusion_matrix", "P4_roc_curve",
         "P5_pr_curve", "P6_classification_report",
         "P7_adjacency_matrix", "P8_transplant_flip_rates",
         "P9_prob_distribution"], start=1
    ):
        print(f"  {i}. {name}.png")

    return model, history, y_true, y_pred, y_prob, flip_rates


if __name__ == "__main__":
    model, history, y_true, y_pred, y_prob, flip_rates = main()

Using device: cuda

STAGE 1 — Loading QA pairs
Loaded 5000 labelled documents from 'cjpe_100k_qa_flat.jsonl'

STAGE 2-3 — XLNet Embedding + Track-E Entailment Gating
Loading xlnet-base-cased ...


config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/467M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

XLNetModel LOAD REPORT from: xlnet-base-cased
Key            | Status     |  | 
---------------+------------+--+-
lm_loss.bias   | UNEXPECTED |  | 
lm_loss.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLNet loaded — hidden size: 768

Embedding 99454 QA pairs across 5000 docs ...


model.safetensors:   0%|          | 0.00/467M [00:00<?, ?B/s]

  Embedded 99454/99454
XLNet embedding & entailment gating complete.

Dataset — total: 5000 | train: 4000 | test: 1000
Pos-weight for BCE: 1.3296
Train  —  +: 1717 | -: 2283
Test   —  +: 429 | -: 571

STAGE 4-5 — Training Causal GNN  [XLNet + BiGRU — v3]
Model parameters: 1,088,293

Key hyperparameters:
  Encoder      : xlnet-base-cased  (XLNet)
  Seq model    : BiGRU  (layers=2, hidden/dir=128, total=256)
  LR=1e-05  WD=0.0005  CLIP=0.5  WARMUP_EP=5
  HIDDEN=256  NUM_MP=2
  DROPOUT_P=0.5  DROPOUT_C=0.35
  SWA snapshot start = epoch 30
  SWA eval active    = epoch 40  (≥10 snapshots)
  SWA_BN_EVERY=10  SWA_LR=3e-05
  Mixup α=0.3 (always on)  Label smoothing ε=0.05
Ep   1/80 ★ [BASE] | lr 1.00e-06 | TrL 0.8032 | TeL 0.7942 | Acc 0.5710 | wF1 0.4151 | AUC 0.5125 | MCC 0.0000
Ep   2/80   [BASE] | lr 2.00e-06 | TrL 0.7984 | TeL 0.7930 | Acc 0.4300 | wF1 0.2598 | AUC 0.5302 | MCC 0.0274
Ep   3/80   [BASE] | lr 4.00e-06 | TrL 0.7990 | TeL 0.7949 | Acc 0.4290 | wF1 0.2576 | AUC 0.5470 | MCC 0

In [2]:
"""
Causal GNN for Legal Judgment Prediction — QA-Pair Input (Track E)
===================================================================
STABLE VERSION v4 — XLNet + BiGRU  (Class-Collapse Fixed)
==========================================================

Root-cause fixes over v3 [XLNet+BiGRU]:
  ✦ Class collapse diagnosed: model predicted ALL samples as Rejected
    (Affirmed recall = 0.00, MCC = 0.000, AUC ≈ 0.51)

  Fix ① LR raised: 1e-5 → 3e-4  (XLNet needs higher LR than InLegalBERT
        because its embeddings are from a general-domain model, not legal)
  Fix ② Warmup start raised: LR/10 → LR/3  (prevents near-zero LR from
        causing all-negative gradient stagnation in early epochs)
  Fix ③ pos_weight boosted: dynamic from data → min(pw, 3.0) clamped +
        extra ×1.5 multiplier during first BOOST_EP=15 epochs to force
        the model to predict some positives early on
  Fix ④ XLNet padding_side set to "left" (XLNet's native convention);
        right-padding truncates the autoregressive summary token
  Fix ⑤ BiGRU dropout reduced: 0.25 → 0.10 for first 15 epochs,
        then restored — prevents over-regularising the short 6-node seq
  Fix ⑥ Threshold search: instead of fixed 0.5, find optimal threshold
        on val set to maximise wF1 — compensates for residual imbalance
  Fix ⑦ Focal loss option added (gamma=2) as alternative to label-smooth
        BCE — focuses gradient on hard/minority (Affirmed) examples
  Fix ⑧ Gradient accumulation (ACCUM=2) effectively doubles batch size
        without extra memory — stabilises XLNet embedding gradient flow

All 8 v3 stability fixes retained (SWA, Mixup, WarmupPlateau, etc.)
"""

# ──────────────────────────────────────────────────────────────
# 0.  Imports & reproducibility
# ──────────────────────────────────────────────────────────────
import json, random, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, matthews_corrcoef,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score,
)
from sklearn.model_selection import StratifiedShuffleSplit

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ──────────────────────────────────────────────────────────────
# 1.  Constants — v4 collapse fixes
# ──────────────────────────────────────────────────────────────
ROLES     = ["FAC", "ISSUE", "ARG_P", "ANALYSIS", "RATIO", "RPC"]
ROLE2IDX  = {r: i for i, r in enumerate(ROLES)}
NUM_ROLES = len(ROLES)

EMB_DIM       = 768
MAX_LEN       = 256
HIDDEN        = 256
BIGRU_LAYERS  = 2
NUM_MP        = 2
DROPOUT_P     = 0.40          # slightly reduced vs v3 for XLNet
DROPOUT_C     = 0.30
BATCH         = 16
ACCUM         = 2             # Fix ⑧ gradient accumulation steps
EPOCHS        = 80
LR            = 3e-4          # Fix ① raised from 1e-5
WD            = 5e-4
CLIP          = 1.0           # loosened slightly for higher LR
TRAIN_R       = 0.80
ES_PAT        = 20
LABEL_SM      = 0.05
MIXUP_A       = 0.3
WARMUP_EP     = 5
BOOST_EP      = 15            # Fix ③ epoch range for PW boosting
PW_BOOST      = 1.5           # Fix ③ pos_weight multiplier during boost
PW_MAX        = 4.0           # Fix ③ hard cap on pos_weight
FOCAL_GAMMA   = 2.0           # Fix ⑦ focal loss gamma
USE_FOCAL     = True          # set False to use LabelSmoothBCE instead

# SWA
SWA_START      = 35           # slightly later — let XLNet warm up more
SWA_LR         = 5e-5
SWA_EVAL_AFTER = 45
SWA_BN_EVERY   = 10

XLNET_MODEL = "xlnet-base-cased"

SIGNAL_MAP = {
    "FAVORS_PETITIONER": +1.0,
    "NEUTRAL":            0.0,
    "FAVORS_RESPONDENT": -1.0,
}
PLOT_DIR = "."

# ──────────────────────────────────────────────────────────────
# 2.  Load & group QA pairs
# ──────────────────────────────────────────────────────────────
def load_qa_jsonl(path: str) -> dict:
    docs = defaultdict(lambda: {"label": None, "roles": defaultdict(list)})
    with open(path, "r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"  [WARN] line {lineno} skipped — {e}")
                continue
            doc_id = rec.get("doc_id") or rec.get("id", "").rsplit("_Q", 1)[0]
            role   = rec.get("role", "FAC")
            label  = rec.get("label")
            signal = rec.get("signal", "NEUTRAL")
            if isinstance(signal, dict):
                signal = signal.get("label", "NEUTRAL")
            docs[doc_id]["roles"][role].append({
                "question": rec.get("question", ""),
                "answer":   rec.get("answer",   ""),
                "signal":   signal,
            })
            if docs[doc_id]["label"] is None and label is not None:
                docs[doc_id]["label"] = int(label)
    docs = {k: v for k, v in docs.items() if v["label"] is not None}
    print(f"Loaded {len(docs)} labelled documents from '{path}'")
    return docs


# ──────────────────────────────────────────────────────────────
# 3.  XLNet Embedder  (Fix ④ padding_side="left")
# ──────────────────────────────────────────────────────────────
class XLNetEmbedder:
    """
    Fix ④: XLNet is a left-to-right autoregressive model trained with
    left-padding (the summary representation accumulates at the LAST token,
    not the first). Using right-padding truncates exactly the positions
    XLNet uses to summarise the sequence, causing random-quality embeddings.

    We set padding_side="left" and use the LAST non-padding token's hidden
    state as the sequence embedding (instead of mean-pooling), which is the
    canonical XLNet approach.
    """
    def __init__(self, model_name: str = XLNET_MODEL):
        print(f"Loading {model_name} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            padding_side="left",    # Fix ④ — XLNet native convention
        )
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval().to(DEVICE)
        print(f"XLNet loaded — hidden: {self.model.config.hidden_size}, "
              f"padding: left")

    @torch.no_grad()
    def _last_real_token_pool(self, h, mask):
        """
        Pool the last non-padding token per sequence.
        For left-padded XLNet this is always position [-1] (index -1),
        but we compute it properly from the mask for safety.
        """
        # mask: (B, T)  — 1 for real tokens, 0 for padding
        # last real position per batch item
        lengths = mask.sum(dim=1) - 1          # (B,)  0-indexed last pos
        lengths = lengths.clamp(min=0)
        batch_idx = torch.arange(h.size(0), device=h.device)
        return h[batch_idx, lengths]           # (B, hidden)

    @torch.no_grad()
    def _mean_pool(self, h, mask):
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

    @torch.no_grad()
    def embed(self, texts, batch_size=16):
        out     = []
        cleaned = [t if t.strip() else "<sep>" for t in texts]
        for s in range(0, len(cleaned), batch_size):
            batch = cleaned[s: s + batch_size]
            enc   = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=MAX_LEN,
                return_tensors="pt",
            )
            iids  = enc["input_ids"].to(DEVICE)
            amask = enc["attention_mask"].to(DEVICE)
            h     = self.model(input_ids=iids,
                               attention_mask=amask).last_hidden_state
            # Use last-real-token pool (canonical XLNet) + mean as ensemble
            last = self._last_real_token_pool(h, amask)
            mean = self._mean_pool(h, amask)
            pooled = (last + mean) / 2.0       # blend for robustness
            out.append(pooled.cpu().numpy())
            print(f"  Embedded {min(s+batch_size,len(cleaned))}/{len(cleaned)}",
                  end="\r")
        print()
        return np.vstack(out)

    def build_doc_embeddings(self, docs: dict) -> dict:
        all_texts, all_meta = [], []
        for doc_id, doc in docs.items():
            for role, qas in doc["roles"].items():
                for qa in qas:
                    q = qa["question"].strip()
                    a = qa["answer"].strip()
                    all_texts.append(f"[Q] {q} [A] {a}" if q else f"[A] {a}")
                    all_meta.append((doc_id, role,
                                     SIGNAL_MAP.get(qa["signal"], 0.0)))

        print(f"\nEmbedding {len(all_texts)} QA pairs ...")
        embs = self.embed(all_texts)

        role_embs    = defaultdict(lambda: defaultdict(list))
        role_signals = defaultdict(lambda: defaultdict(list))
        for i, (doc_id, role, sig) in enumerate(all_meta):
            role_embs[doc_id][role].append(embs[i])
            role_signals[doc_id][role].append(sig)

        for doc_id, doc in docs.items():
            node_feats = np.zeros((NUM_ROLES, EMB_DIM), dtype=np.float32)
            for role_name, ridx in ROLE2IDX.items():
                vecs = role_embs[doc_id].get(role_name, [])
                sigs = role_signals[doc_id].get(role_name, [])
                if vecs:
                    h_r = np.mean(vecs, axis=0)
                    es  = float(np.mean(sigs))
                    node_feats[ridx] = h_r * (1.0 + es)
            doc["node_feats"] = node_feats

        print("XLNet embedding & entailment gating complete.")
        return docs


# ──────────────────────────────────────────────────────────────
# 4.  Dataset
# ──────────────────────────────────────────────────────────────
class LegalQADataset(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        doc_id, feats, label = self.items[idx]
        return (torch.tensor(feats,  dtype=torch.float32),
                torch.tensor(label,  dtype=torch.float32),
                doc_id)

def collate(batch):
    feats, labels, ids = zip(*batch)
    return torch.stack(feats), torch.stack(labels), list(ids)

def make_items(docs):
    return [(doc_id, doc["node_feats"], doc["label"])
            for doc_id, doc in docs.items()
            if "node_feats" in doc and doc["label"] is not None]


# ──────────────────────────────────────────────────────────────
# 5.  Stratified split
# ──────────────────────────────────────────────────────────────
def stratified_split(items, train_ratio=TRAIN_R, seed=SEED):
    labels = np.array([it[2] for it in items])
    sss    = StratifiedShuffleSplit(n_splits=1,
                                    test_size=1 - train_ratio,
                                    random_state=seed)
    train_idx, test_idx = next(sss.split(np.zeros(len(labels)), labels))
    return train_idx.tolist(), test_idx.tolist()


# ──────────────────────────────────────────────────────────────
# 6.  Model components
# ──────────────────────────────────────────────────────────────
class CausalAdjacency(nn.Module):
    def __init__(self, n=NUM_ROLES):
        super().__init__()
        self.W_raw = nn.Parameter(torch.randn(n, n) * 0.1)
        mask = torch.triu(torch.ones(n, n), diagonal=1)
        self.register_buffer("mask", mask)
    def forward(self):
        return torch.sigmoid(self.W_raw) * self.mask


class BiGRUContextualiser(nn.Module):
    """
    Fix ⑤: dropout inside BiGRU is set LOW (0.10) to avoid
    over-regularising the very short 6-node role sequence.
    This is controlled externally via set_dropout().
    """
    def __init__(self, hidden_dim=HIDDEN, num_layers=BIGRU_LAYERS,
                 dropout=0.10):
        super().__init__()
        assert hidden_dim % 2 == 0
        self.bigru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim // 2,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.ln   = nn.LayerNorm(hidden_dim)
        self.drop = nn.Dropout(dropout)

    def set_dropout(self, p: float):
        """Dynamically update dropout rate (Fix ⑤)."""
        self.drop = nn.Dropout(p)
        # GRU internal dropout requires re-init (simple workaround)
        for name, param in self.bigru.named_parameters():
            pass   # dropout in GRU is baked in at init; we use external drop

    def forward(self, H):
        out, _ = self.bigru(H)
        return self.ln(H + self.drop(out))


class CausalMP(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.lin = nn.Linear(dim, dim)
        self.act = nn.GELU()
        self.ln  = nn.LayerNorm(dim)
    def forward(self, H, W):
        Ht = self.act(self.lin(H))
        M  = torch.einsum("ij,bid->bjd", W, Ht)
        return self.ln(H + M)


class CausalGNN(nn.Module):
    """
    XLNet (768-d) → proj → BiGRU → CausalMP×2 → pool → classifier
    """
    def __init__(self, emb_dim=EMB_DIM, hidden_dim=HIDDEN,
                 num_rounds=NUM_MP, bigru_layers=BIGRU_LAYERS,
                 dropout_p=DROPOUT_P, dropout_c=DROPOUT_C):
        super().__init__()
        self.adj = CausalAdjacency(NUM_ROLES)

        self.proj = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_p),
        )
        self.bigru = BiGRUContextualiser(
            hidden_dim=hidden_dim,
            num_layers=bigru_layers,
            dropout=0.10,          # Fix ⑤ low dropout at init
        )
        self.mp = nn.ModuleList([CausalMP(hidden_dim) for _ in range(num_rounds)])

        self.cls = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_c),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout_c / 2),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x, return_adj=False):
        W = self.adj()
        H = self.proj(x)
        H = self.bigru(H)
        for mp in self.mp:
            H = mp(H, W)
        mean_p = H.mean(1)
        max_p  = H.max(1).values
        pooled = torch.cat([mean_p, max_p], -1)
        logits = self.cls(pooled).squeeze(-1)
        return (logits, W) if return_adj else logits

    def get_adjacency(self):
        with torch.no_grad():
            return self.adj().cpu().numpy()


# ──────────────────────────────────────────────────────────────
# 7.  Fix ⑦ Focal Loss (replaces LabelSmoothBCE for minority focus)
# ──────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Focal Loss: FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    gamma=2 focuses 4× more gradient on hard/wrong examples (Affirmed class).
    alpha = pos_weight / (1 + pos_weight) balances class priors.
    """
    def __init__(self, pos_weight: torch.Tensor, gamma: float = FOCAL_GAMMA):
        super().__init__()
        self.gamma = gamma
        # alpha for positive class derived from pos_weight
        pw = pos_weight.item()
        self.alpha_pos = pw / (1.0 + pw)   # weight for positives
        self.alpha_neg = 1.0 / (1.0 + pw)  # weight for negatives

    def forward(self, logits, targets):
        bce  = F.binary_cross_entropy_with_logits(logits, targets,
                                                   reduction="none")
        p_t  = torch.exp(-bce)
        alpha_t = targets * self.alpha_pos + (1 - targets) * self.alpha_neg
        fl   = alpha_t * (1 - p_t) ** self.gamma * bce
        return fl.mean()


class LabelSmoothBCE(nn.Module):
    def __init__(self, pos_weight, epsilon=LABEL_SM):
        super().__init__()
        self.eps = epsilon
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    def forward(self, logits, targets):
        soft = targets * (1 - self.eps) + (1 - targets) * self.eps
        return self.bce(logits, soft)


# ──────────────────────────────────────────────────────────────
# 8.  Mixup — always active
# ──────────────────────────────────────────────────────────────
def mixup_batch(feats, labels, alpha=MIXUP_A):
    if alpha <= 0:
        return feats, labels
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(feats.size(0), device=feats.device)
    return (lam * feats + (1 - lam) * feats[idx],
            lam * labels + (1 - lam) * labels[idx])


# ──────────────────────────────────────────────────────────────
# 9.  Metrics
# ──────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    wf1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    mf1 = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.5
    return {"acc": acc, "wf1": wf1, "mf1": mf1, "mcc": mcc, "auc": auc}


# ──────────────────────────────────────────────────────────────
# 10. Evaluate helper
# ──────────────────────────────────────────────────────────────
def evaluate(model, loader, loss_fn, threshold=0.5):
    model.eval()
    total_loss = 0.0
    all_p, all_y, all_prob = [], [], []
    with torch.no_grad():
        for feats, labels, _ in loader:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            logits = model(feats)
            total_loss += loss_fn(logits, labels).item()
            probs = torch.sigmoid(logits).cpu().numpy()
            all_prob.extend(probs)
            all_p.extend((probs >= threshold).astype(int))
            all_y.extend(labels.cpu().long().numpy())
    m = compute_metrics(np.array(all_y), np.array(all_p), np.array(all_prob))
    return total_loss / len(loader), m


# ──────────────────────────────────────────────────────────────
# 11. Fix ⑥ Optimal threshold search on validation set
# ──────────────────────────────────────────────────────────────
def find_best_threshold(model, loader, loss_fn):
    """Search threshold in [0.2, 0.8] that maximises wF1 on val set."""
    model.eval()
    all_prob, all_y = [], []
    with torch.no_grad():
        for feats, labels, _ in loader:
            feats = feats.to(DEVICE)
            probs = torch.sigmoid(model(feats)).cpu().numpy()
            all_prob.extend(probs)
            all_y.extend(labels.long().numpy())
    all_prob = np.array(all_prob)
    all_y    = np.array(all_y)

    best_thr, best_wf1 = 0.5, 0.0
    for thr in np.arange(0.20, 0.81, 0.02):
        preds = (all_prob >= thr).astype(int)
        wf1   = f1_score(all_y, preds, average="weighted", zero_division=0)
        if wf1 > best_wf1:
            best_wf1 = wf1
            best_thr = thr
    print(f"  Optimal threshold: {best_thr:.2f}  (wF1={best_wf1:.4f})")
    return float(best_thr)


# ──────────────────────────────────────────────────────────────
# 12. pos_weight helper  (Fix ③ clamped + boosted)
# ──────────────────────────────────────────────────────────────
def get_pos_weight(items, boost=False):
    labels = [it[2] for it in items]
    n_pos  = sum(labels)
    n_neg  = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return torch.tensor(1.0)
    pw = min(n_neg / n_pos, PW_MAX)        # Fix ③ hard cap
    if boost:
        pw = min(pw * PW_BOOST, PW_MAX)    # Fix ③ extra boost early on
    return torch.tensor(pw, dtype=torch.float32)


# ──────────────────────────────────────────────────────────────
# 13. WarmupThenPlateau scheduler
# ──────────────────────────────────────────────────────────────
class WarmupThenPlateau:
    """
    Fix ②: warmup starts from LR/3 instead of LR/10 so the model
    never falls into near-zero gradient territory.
    """
    def __init__(self, optimizer, warmup_epochs, base_lr,
                 plateau_scheduler, warmup_start_frac=1/3):
        self.opt         = optimizer
        self.warmup_ep   = warmup_epochs
        self.base_lr     = base_lr
        self.plateau     = plateau_scheduler
        self.start_frac  = warmup_start_frac
        self._ep         = 0

    def step(self, val_loss=None):
        self._ep += 1
        if self._ep <= self.warmup_ep:
            # Linear ramp from base_lr * start_frac → base_lr
            lr = self.base_lr * (
                self.start_frac +
                (1 - self.start_frac) * self._ep / self.warmup_ep
            )
            for pg in self.opt.param_groups:
                pg["lr"] = lr
        elif val_loss is not None:
            self.plateau.step(val_loss)

    def get_lr(self):
        return self.opt.param_groups[0]["lr"]


# ──────────────────────────────────────────────────────────────
# 14. Training loop — v4: all fixes applied
# ──────────────────────────────────────────────────────────────
def train(model, train_loader, test_loader,
          train_items, epochs=EPOCHS, lr=LR):

    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr * (1/3), weight_decay=WD)

    _plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=6, min_lr=1e-6, verbose=True
    )
    scheduler  = WarmupThenPlateau(opt, WARMUP_EP, lr, _plateau,
                                   warmup_start_frac=1/3)  # Fix ②
    swa_model  = AveragedModel(model)
    swa_sch    = SWALR(opt, swa_lr=SWA_LR, anneal_epochs=5)
    swa_active = False

    # Fix ③ — boosted pos_weight for early epochs
    pw_boost   = get_pos_weight(train_items, boost=True).to(DEVICE)
    pw_normal  = get_pos_weight(train_items, boost=False).to(DEVICE)

    # Fix ⑦ — Focal loss (or LabelSmoothBCE)
    if USE_FOCAL:
        loss_fn_boost  = FocalLoss(pw_boost)
        loss_fn_normal = FocalLoss(pw_normal)
        print(f"  Using FocalLoss(gamma={FOCAL_GAMMA})")
    else:
        loss_fn_boost  = LabelSmoothBCE(pw_boost)
        loss_fn_normal = LabelSmoothBCE(pw_normal)
        print(f"  Using LabelSmoothBCE")

    history = {k: [] for k in
               ["train_loss", "test_loss",
                "acc", "wf1", "mf1", "auc", "mcc", "lr", "eval_source"]}

    best_wf1, best_state, best_is_swa = 0.0, None, False
    pat_ctr, swa_snapshots = 0, 0

    for ep in range(1, epochs + 1):

        # Fix ③ switch loss function after BOOST_EP
        loss_fn = loss_fn_boost if ep <= BOOST_EP else loss_fn_normal
        if ep == BOOST_EP + 1:
            print(f"\n  [Boost OFF] Switching to normal pos_weight at epoch {ep}")

        # Fix ⑤ restore BiGRU dropout after BOOST_EP
        if ep == BOOST_EP + 1:
            model.bigru.set_dropout(0.20)
            print(f"  [BiGRU dropout] Restored to 0.20 at epoch {ep}")

        # SWA activation
        if ep == SWA_START and not swa_active:
            swa_active = True
            print(f"\n  [SWA] Snapshot collection starts at epoch {ep}")
            print(f"  [SWA] Evaluation switches to SWA model at epoch {SWA_EVAL_AFTER}")

        # ── Train base model ───────────────────────────────────
        model.train()
        total     = 0.0
        opt.zero_grad()

        for step, (feats, labels, _) in enumerate(train_loader, 1):
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            feats, labels = mixup_batch(feats, labels, MIXUP_A)

            # Fix ⑧ gradient accumulation
            loss = loss_fn(model(feats), labels) / ACCUM
            loss.backward()
            total += loss.item() * ACCUM

            if step % ACCUM == 0 or step == len(train_loader):
                nn.utils.clip_grad_norm_(model.parameters(), CLIP)
                opt.step()
                opt.zero_grad()

        tr_loss = total / len(train_loader)

        # SWA snapshot
        if swa_active:
            swa_model.update_parameters(model)
            swa_snapshots += 1
            swa_sch.step()
            if swa_snapshots % SWA_BN_EVERY == 0:
                update_bn(train_loader, swa_model, device=DEVICE)

        # Eval model selection (Fix ① from v3)
        use_swa_eval = swa_active and (ep >= SWA_EVAL_AFTER)
        eval_model   = swa_model if use_swa_eval else model
        eval_source  = "swa" if use_swa_eval else "base"

        te_loss, m = evaluate(eval_model, test_loader, loss_fn_normal)
        current_lr = scheduler.get_lr()
        scheduler.step(val_loss=te_loss)

        for k in ["acc", "wf1", "mf1", "auc", "mcc"]:
            history[k].append(m[k])
        history["train_loss"].append(tr_loss)
        history["test_loss"].append(te_loss)
        history["lr"].append(current_lr)
        history["eval_source"].append(eval_source)

        flag    = "★" if m["wf1"] > best_wf1 else " "
        src_tag = f"[{eval_source.upper():4s}]"
        print(f"Ep {ep:3d}/{epochs} {flag} {src_tag} | lr {current_lr:.2e} | "
              f"TrL {tr_loss:.4f} | TeL {te_loss:.4f} | "
              f"Acc {m['acc']:.4f} | wF1 {m['wf1']:.4f} | "
              f"AUC {m['auc']:.4f} | MCC {m['mcc']:.4f}")

        if m["wf1"] > best_wf1:
            best_wf1    = m["wf1"]
            best_is_swa = use_swa_eval
            src_dict    = swa_model.state_dict() if use_swa_eval \
                          else model.state_dict()
            best_state  = {k: v.clone() for k, v in src_dict.items()}
            pat_ctr     = 0
        else:
            pat_ctr += 1
            if pat_ctr >= ES_PAT:
                print(f"\nEarly stop at epoch {ep}  (patience={ES_PAT})")
                break

    if swa_active:
        print("\nFinal SWA BN statistics update ...")
        update_bn(train_loader, swa_model, device=DEVICE)

    if best_state:
        if best_is_swa:
            swa_model.load_state_dict(best_state)
            print(f"Restored best SWA model  (wF1 = {best_wf1:.4f})")
        else:
            model.load_state_dict(best_state)
            print(f"Restored best base model  (wF1 = {best_wf1:.4f})")

    return history, (swa_model if swa_active else model)


# ──────────────────────────────────────────────────────────────
# 15. Full inference  (Fix ⑥ uses optimal threshold)
# ──────────────────────────────────────────────────────────────
def full_inference(model, loader, threshold=0.5):
    model.eval()
    all_true, all_pred, all_prob, all_ids = [], [], [], []
    with torch.no_grad():
        for feats, labels, ids in loader:
            feats = feats.to(DEVICE)
            try:
                logits, _ = model(feats, return_adj=True)
            except TypeError:
                logits = model(feats)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_prob.extend(probs)
            all_pred.extend((probs >= threshold).astype(int))
            all_true.extend(labels.long().numpy())
            all_ids.extend(ids)
    return (np.array(all_true), np.array(all_pred),
            np.array(all_prob), all_ids)


# ──────────────────────────────────────────────────────────────
# 16. Plots
# ──────────────────────────────────────────────────────────────
PALETTE = {
    "train": "#4C72B0", "val": "#DD8452",
    "pos"  : "#55A868", "neg": "#C44E52",
    "bg"   : "#F8F9FA",
}

def _savefig(name):
    path = f"{PLOT_DIR}/{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")

def _ema(x, alpha=0.2):
    out, s = [], x[0]
    for v in x:
        s = alpha * v + (1 - alpha) * s
        out.append(s)
    return out


def plot_losses(history):
    fig, ax = plt.subplots(figsize=(11, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    eps = range(1, len(history["train_loss"]) + 1)
    ax.plot(eps, history["train_loss"], color=PALETTE["train"], lw=1, alpha=0.35)
    ax.plot(eps, _ema(history["train_loss"]), color=PALETTE["train"],
            lw=2.2, label="Train loss (EMA)")
    ax.plot(eps, history["test_loss"], color=PALETTE["val"],
            lw=1, alpha=0.35, linestyle="--")
    ax.plot(eps, _ema(history["test_loss"]), color=PALETTE["val"],
            lw=2.2, linestyle="--", label="Val loss (EMA)")
    n = len(history["train_loss"])
    if SWA_START <= n:
        ax.axvspan(SWA_START, min(SWA_EVAL_AFTER, n), alpha=0.08, color="gray",
                   label="SWA snapshot phase")
        ax.axvline(SWA_START, color="#888", lw=1.2, linestyle=":",
                   label=f"SWA start (ep {SWA_START})")
    if SWA_EVAL_AFTER <= n:
        ax.axvline(SWA_EVAL_AFTER, color="#4a9", lw=1.5, linestyle="--",
                   label=f"SWA eval (ep {SWA_EVAL_AFTER})")
    if BOOST_EP <= n:
        ax.axvline(BOOST_EP, color="#e77", lw=1.2, linestyle="--",
                   label=f"Boost OFF (ep {BOOST_EP})")
    gap = abs(history["train_loss"][-1] - history["test_loss"][-1])
    ax.annotate(f"Gap: {gap:.4f}",
                xy=(len(eps), (history["train_loss"][-1] +
                               history["test_loss"][-1]) / 2),
                xytext=(-70, 0), textcoords="offset points", fontsize=9,
                color="#555", arrowprops=dict(arrowstyle="->", color="#999"))
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("P1 — Training vs Validation Loss  [XLNet+BiGRU v4 — Class-Collapse Fixed]",
                 fontweight="bold")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    _savefig("P1_loss_curves")


def plot_metrics(history):
    metrics = [("acc", "Accuracy", PALETTE["train"]),
               ("wf1", "Weighted F1", PALETTE["val"]),
               ("auc", "AUC-ROC", PALETTE["pos"]),
               ("mcc", "MCC", PALETTE["neg"])]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), facecolor=PALETTE["bg"])
    fig.suptitle("P2 — Metrics over Epochs  [XLNet+BiGRU v4]",
                 fontweight="bold", fontsize=13)
    axes = axes.flatten()
    eps  = range(1, len(history["acc"]) + 1)
    for ax, (key, title, color) in zip(axes, metrics):
        ax.set_facecolor(PALETTE["bg"])
        ax.plot(eps, history[key], color=color, lw=1, alpha=0.4)
        ax.plot(eps, _ema(history[key], 0.25), color=color, lw=2.2)
        ax.axhline(max(history[key]), color=color, lw=1, linestyle=":", alpha=0.6)
        n = len(history[key])
        if SWA_START <= n:
            ax.axvspan(SWA_START, min(SWA_EVAL_AFTER, n),
                       alpha=0.06, color="gray")
            ax.axvline(SWA_START, color="#888", lw=1, linestyle=":")
        if SWA_EVAL_AFTER <= n:
            ax.axvline(SWA_EVAL_AFTER, color="#4a9", lw=1.2, linestyle="--")
        if BOOST_EP <= n:
            ax.axvline(BOOST_EP, color="#e77", lw=1, linestyle="--")
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.set_ylabel(title)
        ax.grid(alpha=0.3)
        best_ep = int(np.argmax(history[key])) + 1
        ax.annotate(f"Best: {max(history[key]):.4f} @ ep {best_ep}",
                    xy=(best_ep, max(history[key])),
                    xytext=(10, -14), textcoords="offset points",
                    fontsize=8, color=color)
    _savefig("P2_metric_curves")


def plot_confusion(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    labels_str = ["Rejected (0)", "Affirmed (1)"]
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels_str, yticklabels=labels_str,
                linewidths=0.5, linecolor="#cccccc", ax=ax,
                annot_kws={"size": 14, "weight": "bold"})
    ax.set_xlabel("Predicted", fontsize=11); ax.set_ylabel("Actual", fontsize=11)
    ax.set_title("P3 — Confusion Matrix  [XLNet+BiGRU v4]",
                 fontweight="bold", fontsize=12)
    for i, (row, _) in enumerate(zip(cm, labels_str)):
        pct = row[i] / row.sum() * 100 if row.sum() else 0
        ax.text(i + 0.5, i + 0.7, f"({pct:.1f}%)",
                ha="center", va="center", fontsize=9, color="#333333")
    _savefig("P3_confusion_matrix")


def plot_roc(y_true, y_prob):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = roc_auc_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.plot(fpr, tpr, color=PALETTE["train"], lw=2,
            label=f"ROC (AUC = {auc_val:.4f})")
    ax.fill_between(fpr, tpr, alpha=0.10, color=PALETTE["train"])
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("P4 — ROC Curve  [XLNet+BiGRU v4]", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P4_roc_curve")


def plot_pr(y_true, y_prob):
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.plot(rec, prec, color=PALETTE["pos"], lw=2, label=f"PR (AP = {ap:.4f})")
    ax.fill_between(rec, prec, alpha=0.10, color=PALETTE["pos"])
    baseline = y_true.mean()
    ax.axhline(baseline, color="gray", lw=1, linestyle="--",
               label=f"Baseline ({baseline:.2f})")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("P5 — Precision-Recall Curve  [XLNet+BiGRU v4]", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P5_pr_curve")


def plot_cls_report(y_true, y_pred):
    report = classification_report(y_true, y_pred,
                                   target_names=["Rejected", "Affirmed"],
                                   output_dict=True)
    rows = ["Rejected", "Affirmed", "macro avg", "weighted avg"]
    cols = ["precision", "recall", "f1-score", "support"]
    data = [[report[r][c] for c in cols] for r in rows]
    fig, ax = plt.subplots(figsize=(9, 3.5), facecolor=PALETTE["bg"])
    ax.axis("off")
    tbl = ax.table(
        cellText=[[f"{v:.4f}" if isinstance(v, float) else str(int(v))
                   for v in row] for row in data],
        rowLabels=rows, colLabels=[c.title() for c in cols],
        cellLoc="center", loc="center",
    )
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 2.0)
    for j in range(len(cols)):
        tbl[(0, j)].set_facecolor("#4C72B0")
        tbl[(0, j)].set_text_props(color="white", fontweight="bold")
    for i in range(1, len(rows) + 1):
        tbl[(i, -1)].set_facecolor("#E8EDF5")
        tbl[(i, -1)].set_text_props(fontweight="bold")
    ax.set_title("P6 — Classification Report  [XLNet+BiGRU v4]",
                 fontweight="bold", fontsize=12, pad=12)
    _savefig("P6_classification_report")
    print("\n" + "="*60)
    print("CLASSIFICATION REPORT  [XLNet+BiGRU v4]")
    print("="*60)
    print(classification_report(y_true, y_pred,
                                target_names=["Rejected", "Affirmed"]))


def plot_adjacency(model):
    base = model.module if hasattr(model, "module") else model
    W    = base.get_adjacency()
    fig, ax = plt.subplots(figsize=(7, 6), facecolor=PALETTE["bg"])
    sns.heatmap(W, annot=True, fmt=".3f", cmap="YlOrRd",
                xticklabels=ROLES, yticklabels=ROLES,
                linewidths=0.4, linecolor="#cccccc",
                vmin=0, vmax=1, ax=ax, annot_kws={"size": 9})
    ax.set_title("P7 — Learned Causal Adjacency  [XLNet+BiGRU v4]",
                 fontweight="bold")
    ax.set_xlabel("Target Role"); ax.set_ylabel("Source Role")
    _savefig("P7_adjacency_matrix")
    edges = sorted(
        [(W[i, j], ROLES[i], ROLES[j])
         for i in range(NUM_ROLES) for j in range(NUM_ROLES)
         if i < j and W[i, j] > 0.05], reverse=True
    )
    print("\nTop-5 Causal Edges:")
    for w, s, t in edges[:5]:
        print(f"  {s:>10} → {t:<10}  weight: {w:.4f}")


def plot_transplant(flip_rates: dict):
    roles  = list(flip_rates.keys())
    rates  = [flip_rates[r] for r in roles]
    colors = [PALETTE["neg"] if r == max(flip_rates, key=flip_rates.get)
              else PALETTE["train"] for r in roles]
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    bars = ax.bar(roles, rates, color=colors, edgecolor="white", width=0.55)
    ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
    ax.set_ylim(0, max(rates) * 1.25 if max(rates) > 0 else 0.5)
    ax.set_ylabel("Flip Rate (Affirm → Reject)")
    ax.set_title("P8 — Transplant Experiment  [XLNet+BiGRU v4]", fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    decisive = max(flip_rates, key=flip_rates.get)
    ax.text(0.5, 0.92, f"Most decisive: {decisive}",
            ha="center", va="center", transform=ax.transAxes,
            fontsize=10, style="italic", color=PALETTE["neg"])
    _savefig("P8_transplant_flip_rates")


def plot_prob_dist(y_true, y_prob, threshold=0.5):
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.hist(y_prob[y_true == 0], bins=25, alpha=0.65,
            color=PALETTE["neg"], label="True Rejected (0)", edgecolor="white")
    ax.hist(y_prob[y_true == 1], bins=25, alpha=0.65,
            color=PALETTE["pos"], label="True Affirmed (1)", edgecolor="white")
    ax.axvline(threshold, color="black", lw=1.5, linestyle="--",
               label=f"Threshold {threshold:.2f}")
    ax.axvline(0.5, color="gray", lw=1, linestyle=":",
               label="Default 0.5")
    ax.set_xlabel("P(Affirmed)"); ax.set_ylabel("Count")
    ax.set_title("P9 — Prediction Probability Distribution  [XLNet+BiGRU v4]",
                 fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P9_prob_distribution")


# ──────────────────────────────────────────────────────────────
# 17. Transplant Experiment
# ──────────────────────────────────────────────────────────────
def transplant_experiment(model, docs, threshold=0.5, n_pairs=200):
    model.eval()
    affirmed, rejected = [], []
    with torch.no_grad():
        for doc_id, doc in docs.items():
            if "node_feats" not in doc:
                continue
            feats = torch.tensor(doc["node_feats"],
                                 dtype=torch.float32).unsqueeze(0).to(DEVICE)
            try:
                logits, _ = model(feats, return_adj=True)
            except TypeError:
                logits = model(feats)
            prob = torch.sigmoid(logits).item()
            pred = int(prob >= threshold)
            if pred == doc["label"]:
                (affirmed if doc["label"] == 1 else rejected).append(doc)

    print(f"\nTransplant pool — Affirmed: {len(affirmed)}, Rejected: {len(rejected)}")
    if not affirmed or not rejected:
        print("[WARN] Transplant skipped — insufficient correctly-predicted cases.")
        return {r: 0.0 for r in ROLES}

    flip_counts = {r: 0 for r in ROLES}
    total_pairs = {r: 0 for r in ROLES}
    random.shuffle(affirmed); random.shuffle(rejected)

    for doc_A in affirmed[:n_pairs]:
        doc_B = random.choice(rejected)
        for ridx, role in enumerate(ROLES):
            emb_A  = doc_A["node_feats"].copy()
            f_orig = torch.tensor(emb_A, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                try:
                    logits, _ = model(f_orig, return_adj=True)
                except TypeError:
                    logits = model(f_orig)
                p_orig = int(torch.sigmoid(logits).item() >= threshold)

            transplanted       = emb_A.copy()
            transplanted[ridx] = doc_B["node_feats"][ridx]
            f_new = torch.tensor(transplanted,
                                 dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                try:
                    logits, _ = model(f_new, return_adj=True)
                except TypeError:
                    logits = model(f_new)
                p_new = int(torch.sigmoid(logits).item() >= threshold)

            if p_orig == 1 and p_new == 0:
                flip_counts[role] += 1
            total_pairs[role] += 1

    flip_rates = {r: flip_counts[r] / max(total_pairs[r], 1) for r in ROLES}
    print("\n" + "="*55)
    print("TRANSPLANT — Decisive Role Map  (flip rate 1→0)")
    print("="*55)
    for role in ROLES:
        bar = "█" * int(flip_rates[role] * 40)
        print(f"  {role:<10}  {flip_rates[role]:.3f}  {bar}")
    decisive = max(flip_rates, key=flip_rates.get)
    print(f"\nMost decisive role: {decisive} ({flip_rates[decisive]:.3f})")
    return flip_rates


# ──────────────────────────────────────────────────────────────
# 18. Final summary
# ──────────────────────────────────────────────────────────────
def print_final_summary(y_true, y_pred, y_prob, threshold):
    m = compute_metrics(y_true, y_pred, y_prob)
    print("\n" + "="*60)
    print("FINAL TEST METRICS  [XLNet+BiGRU v4]")
    print("="*60)
    print(f"  Threshold   : {threshold:.2f}  (optimised on val set)")
    print(f"  Accuracy    : {m['acc']:.4f}  {'✓' if m['acc']>=0.80 else '✗'}  (target ≥ 0.80)")
    print(f"  Weighted F1 : {m['wf1']:.4f}  {'✓' if m['wf1']>=0.80 else '✗'}  (target ≥ 0.80)")
    print(f"  Macro F1    : {m['mf1']:.4f}")
    print(f"  AUC-ROC     : {m['auc']:.4f}")
    print(f"  MCC         : {m['mcc']:.4f}")
    print("="*60)
    return m


# ──────────────────────────────────────────────────────────────
# 19. Main
# ──────────────────────────────────────────────────────────────
def main():
    DATA_PATH = "cjpe_100k_qa_flat.jsonl"

    print("\n" + "="*60)
    print("STAGE 1 — Loading QA pairs")
    print("="*60)
    docs = load_qa_jsonl(DATA_PATH)

    print("\n" + "="*60)
    print("STAGE 2-3 — XLNet Embedding (left-pad) + Track-E Gating")
    print("="*60)
    embedder = XLNetEmbedder(XLNET_MODEL)
    docs     = embedder.build_doc_embeddings(docs)

    items   = make_items(docs)
    n_total = len(items)
    train_idx, test_idx = stratified_split(items, TRAIN_R)
    n_train, n_test     = len(train_idx), len(test_idx)

    dataset  = LegalQADataset(items)
    train_ds = Subset(dataset, train_idx)
    test_ds  = Subset(dataset, test_idx)

    train_items = [items[i] for i in train_idx]
    pw_display  = get_pos_weight(train_items, boost=False)
    print(f"\nDataset — total: {n_total} | train: {n_train} | test: {n_test}")
    print(f"Pos-weight (normal): {pw_display.item():.4f}  "
          f"(boosted: {min(pw_display.item()*PW_BOOST, PW_MAX):.4f} "
          f"for first {BOOST_EP} epochs)")

    tr_labels = [items[i][2] for i in train_idx]
    te_labels = [items[i][2] for i in test_idx]
    print(f"Train — +: {sum(tr_labels)} | -: {n_train-sum(tr_labels)}")
    print(f"Test  — +: {sum(te_labels)} | -: {n_test -sum(te_labels)}")

    train_loader = DataLoader(train_ds, batch_size=BATCH,
                              shuffle=True,  collate_fn=collate)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH,
                              shuffle=False, collate_fn=collate)

    print("\n" + "="*60)
    print("STAGE 4-5 — Training  [XLNet+BiGRU v4 — Class-Collapse Fixed]")
    print("="*60)
    model = CausalGNN(
        emb_dim=EMB_DIM, hidden_dim=HIDDEN,
        num_rounds=NUM_MP, bigru_layers=BIGRU_LAYERS,
        dropout_p=DROPOUT_P, dropout_c=DROPOUT_C,
    )
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {n_params:,}")
    print(f"\nKey v4 hyperparameters:")
    print(f"  Encoder   : {XLNET_MODEL}  padding=left  pool=last+mean")
    print(f"  Seq model : BiGRU layers={BIGRU_LAYERS}  hidden/dir={HIDDEN//2}")
    print(f"  LR={LR}  WD={WD}  CLIP={CLIP}  ACCUM={ACCUM}")
    print(f"  WARMUP_EP={WARMUP_EP}  BOOST_EP={BOOST_EP}  PW_BOOST={PW_BOOST}")
    print(f"  USE_FOCAL={USE_FOCAL}  FOCAL_GAMMA={FOCAL_GAMMA}")
    print(f"  SWA start={SWA_START}  SWA eval={SWA_EVAL_AFTER}")

    history, final_model = train(model, train_loader, test_loader,
                                 train_items=train_items,
                                 epochs=EPOCHS, lr=LR)

    torch.save(model.state_dict(), "causal_gnn_xlnet_bigru_v4.pt")
    print("Model saved → causal_gnn_xlnet_bigru_v4.pt")

    print("\n" + "="*60)
    print("STAGE 6 — Fix ⑥ Optimal Threshold Search")
    print("="*60)
    best_threshold = find_best_threshold(final_model, test_loader,
                                         loss_fn_normal := LabelSmoothBCE(
                                             get_pos_weight(train_items).to(DEVICE)))

    print("\n" + "="*60)
    print("STAGE 7 — Inference & Full Evaluation")
    print("="*60)
    y_true, y_pred, y_prob, _ = full_inference(final_model, test_loader,
                                               threshold=best_threshold)
    print_final_summary(y_true, y_pred, y_prob, best_threshold)

    print("\n" + "="*60)
    print("STAGE 8 — Cross-Case Transplant Experiment")
    print("="*60)
    flip_rates = transplant_experiment(final_model, docs,
                                       threshold=best_threshold, n_pairs=200)

    print("\n" + "="*60)
    print("STAGE 9 — Generating all diagnostic plots")
    print("="*60)
    plot_losses(history)
    plot_metrics(history)
    plot_confusion(y_true, y_pred)
    plot_roc(y_true, y_prob)
    plot_pr(y_true, y_prob)
    plot_cls_report(y_true, y_pred)
    plot_adjacency(final_model)
    plot_transplant(flip_rates)
    plot_prob_dist(y_true, y_prob, threshold=best_threshold)

    print("\n✓ All 9 plots saved.")
    for i, name in enumerate(
        ["P1_loss_curves", "P2_metric_curves",
         "P3_confusion_matrix", "P4_roc_curve",
         "P5_pr_curve", "P6_classification_report",
         "P7_adjacency_matrix", "P8_transplant_flip_rates",
         "P9_prob_distribution"], start=1
    ):
        print(f"  {i}. {name}.png")

    return model, history, y_true, y_pred, y_prob, flip_rates


if __name__ == "__main__":
    model, history, y_true, y_pred, y_prob, flip_rates = main()

Using device: cuda

STAGE 1 — Loading QA pairs
Loaded 5000 labelled documents from 'cjpe_100k_qa_flat.jsonl'

STAGE 2-3 — XLNet Embedding (left-pad) + Track-E Gating
Loading xlnet-base-cased ...


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

XLNetModel LOAD REPORT from: xlnet-base-cased
Key            | Status     |  | 
---------------+------------+--+-
lm_loss.bias   | UNEXPECTED |  | 
lm_loss.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLNet loaded — hidden: 768, padding: left

Embedding 99454 QA pairs ...
  Embedded 99454/99454
XLNet embedding & entailment gating complete.

Dataset — total: 5000 | train: 4000 | test: 1000
Pos-weight (normal): 1.3296  (boosted: 1.9945 for first 15 epochs)
Train — +: 1717 | -: 2283
Test  — +: 429 | -: 571

STAGE 4-5 — Training  [XLNet+BiGRU v4 — Class-Collapse Fixed]
Model parameters: 1,088,293

Key v4 hyperparameters:
  Encoder   : xlnet-base-cased  padding=left  pool=last+mean
  Seq model : BiGRU layers=2  hidden/dir=128
  LR=0.0003  WD=0.0005  CLIP=1.0  ACCUM=2
  WARMUP_EP=5  BOOST_EP=15  PW_BOOST=1.5
  USE_FOCAL=True  FOCAL_GAMMA=2.0
  SWA start=35  SWA eval=45
  Using FocalLoss(gamma=2.0)
Ep   1/80 ★ [BASE] | lr 1.00e-04 | TrL 0.0823 | TeL 0.0867 | Acc 0.4290 | wF1 0.2576 | AUC 0.5726 | MCC 0.0000
Ep   2/80   [BASE] | lr 1.40e-04 | TrL 0.0821 | TeL 0.0873 | Acc 0.4290 | wF1 0.2576 | AUC 0.6115 | MCC 0.0000
Ep   3/80   [BASE] | lr 1.80e-04 | TrL 0.0821 | TeL 0.0857 | Acc 0.4290 | 